In [10]:
import os
import time
import logging
import traceback
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from urllib.parse import urlparse, urlunparse
from rapidfuzz import fuzz

Metadata=pd.read_excel(r"C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\Hoosier_WebScraping_Output-Metadata_All_Pages.xlsx")
print(Metadata.shape)
Metadata = Metadata.loc[(Metadata["Consider for Detailed Scraping"] == "Y") & (Metadata["Is in Indiana State?"] == "Yes"), ["Employer Name", "City"]].drop_duplicates()
Metadata['Employer Name'] = Metadata['Employer Name'].str.replace('#', '', regex=False)
print(Metadata.shape)

(227157, 8)
(21041, 2)


#### trial for fuzzy matching

In [9]:
# Setup logging
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True)  # Force logging to overwrite any previous settings

# Load credentials from environment variables (recommended)
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

# Setup Chrome options
options = Options()
# options.add_argument("--headless")  # Uncomment if running in headless mode
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# Step 1: Open Capital IQ login page
driver.get("https://www.capitaliq.spglobal.com")
logging.info("Opened Capital IQ login page.")

# Step 2: Enter email
email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
email_input.clear()
email_input.send_keys(EMAIL)
logging.info("Email entered.")

# Step 3: Click "Next"
next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]')))
next_button.click()
logging.info("Email submitted.")

# Step 4: Enter password
password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
password_input.send_keys(PASSWORD)

# Step 5: Click "Sign In"
sign_in_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]')))
sign_in_button.click()
logging.info("Password submitted.")

# Step 6: Wait for login to complete
wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "header")]')))
logging.info("Login successful and page loaded.")

# Step 7: Accept cookie popup if present
try:
    cookie_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
    cookie_button.click()
    logging.info("Cookie popup accepted.")
except:
    logging.info("No cookie popup found or already handled.")

# Step 8: Navigate directly to the search results URL
all_company_data = []

for index, row in Metadata.iterrows():
    search_term = row['Employer Name']
    search_city = row['City']
    logging.info(f"Processing: {search_term} in {search_city}")

    search_url = f"https://www.capitaliq.spglobal.com/apisv3/spg-webplatform-core/search/searchResults?vertical=private_companies_mi-gss&q={search_term.replace(' ', '%20')}"
    driver.get(search_url)
    logging.info(f"Navigated to search results for: {search_term}")

    # Step 8: Check if search results loaded
    try:
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))
    except:
        logging.warning(f"No Search results for {search_term}, continuing to next company")
        company_data = {
            "Employer Name": search_term,
            "City": search_city,
            "NAICS Code": None,
            "Industry": None,
            "Description": None,
            "Website URL": None,
            "Headquarters": None,
            "Current_Headcount": None,
            "Percent_Change_in_HC": None,
            "Change_in_HC": None,
            "CEO Name": None,
            "CEO Profile Link": None,
            "LinkedIN URL": None,
            "Key People": None
        }
        all_company_data.append(company_data)
        continue

    # Step 8.1: Handle cookie popup
    try:
        cookie_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
        cookie_button.click()
        logging.info("Cookie popup accepted.")
    except:
        logging.info("No cookie popup found or already handled.")

    # Step 9: Change results per page to 50
    try:
        show_label = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))
        driver.execute_script("arguments[0].scrollIntoView(true);", show_label)
        time.sleep(1)

        dropdown_trigger = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@type="button" and @title="20"]')))
        driver.execute_script("arguments[0].click();", dropdown_trigger)
        time.sleep(1)

        options = wait.until(EC.presence_of_all_elements_located((By.XPATH, '//div[@title="50" and contains(@class, "css-11d71qm")]')))
        for option in options:
            if option.is_displayed():
                driver.execute_script("arguments[0].click();", option)
                logging.info("✅ Changed results per page to 50")
                time.sleep(3)
                break
    except Exception as e:
        logging.error(f"Error while changing results per page: {e}")

    # Step 10: Find matching company by city
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]'))
        )
        company_links = driver.find_elements(By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]')
        match_found = False

        employer_input = search_term.lower()
        for link in company_links:
            try:
                container = link.find_element(By.XPATH, './ancestor::div[3]')
                container_text = container.text.lower()
                company_name = link.text.strip().lower()
                city_match = search_city.lower() in container_text
                name_similarity = fuzz.token_sort_ratio(employer_input, company_name)

                if city_match and name_similarity >= 80:
                    logging.info(f"✅ Match found: {link.text.strip()} (Similarity: {name_similarity}%)")
                    driver.execute_script("arguments[0].click();", link)
                    match_found = True
                    # Step 11: Initialize company_data for matched company
                    company_data = {
                        "Employer Name": search_term,
                        "City": search_city,
                        "NAICS Code": None,
                        "Industry": None,
                        "Description": None,
                        "Website URL": None,
                        "Headquarters": None,
                        "Current_Headcount": None,
                        "Percent_Change_in_HC": None,
                        "Change_in_HC": None,
                        "CEO Name": None,
                        "CEO Profile Link": None,
                        "LinkedIN URL": None,
                        "Key People": None
                    }
                    # Continue with data extraction steps...

                    # 1. NAICS Code
                    try:
                        naics_element = wait.until(EC.presence_of_element_located(
                            (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
                        company_data["NAICS Code"] = naics_element.text.strip()
                    except Exception as e:
                        print("NAICS Code not found:")

                    # 2. Industry
                    try:
                        industry_element = wait.until(EC.presence_of_element_located(
                            (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
                        company_data["Industry"] = industry_element.text.strip()
                    except Exception as e:
                        print("Industry not found!")

                    # 3. Description
                    try:
                        view_all_button = wait.until(EC.element_to_be_clickable(
                            (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
                        view_all_button.click()
                        time.sleep(1)
                        description_element = wait.until(EC.presence_of_element_located((By.ID, "compDesc")))
                        description = description_element.text.strip()
                        if "VIEW LESS" in description:
                            description = description.split("VIEW LESS")[0].strip()
                        company_data["Description"] = description
                    except Exception as e:
                        print("Description not found!")

                    # 4. Website URL
                    try:
                        cpweblink_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cpweblink")))
                        company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")
                    except Exception as e:
                        print("Website URL not found!")

                    # 5. Headquarters
                    try:
                        block_label_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "blockLabel")))
                        hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
                        lines = hq_element.text.splitlines()
                        company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])
                    except Exception as e:
                        print("Headquarters not found!")

                    # 6. Current Headcount
                    try:
                        headcount_element = wait.until(EC.presence_of_element_located((
                            By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
                        company_data["Current_Headcount"] = headcount_element.text.strip()
                    except Exception as e:
                        print("Current Headcount not found!")

                    # 7. Percent Change in Headcount
                    try:
                        percent_change = wait.until(EC.presence_of_element_located((By.XPATH, '//strong[@class="spg-ml-xs"]')))
                        company_data["Percent_Change_in_HC"] = percent_change.text.strip().replace('%', '')
                    except Exception as e:
                        print("Percent Change in Headcount not found!")

                    # 8. Change in Headcount Direction
                    try:
                        change_type = wait.until(EC.presence_of_element_located((By.XPATH,
                            '//div[contains(@class, "spg-d-flex") and contains(@class, "spg-align-center")]//span[@data-icon]')))
                        icon_html = change_type.get_attribute('outerHTML')
                        if 'caret-up' in icon_html:
                            company_data["Change_in_HC"] = 1
                        elif 'caret-down' in icon_html:
                            company_data["Change_in_HC"] = -1
                        else:
                            company_data["Change_in_HC"] = 0
                    except Exception as e:
                        print("Change in Headcount direction not found!")

                    # 9. CEO Name and Profile Link
                    try:
                        ceo_element = wait.until(EC.presence_of_element_located((
                            By.CSS_SELECTOR, 'a[data-testid="ribbon-ceo"].spg-text-medium.css-12tcig2.css-1b9i013')))
                        company_data["CEO Name"] = ceo_element.text
                        company_data["CEO Profile Link"] = ceo_element.get_attribute("href")
                    except Exception as e:
                        print("CEO information not found!")

                    # 11. Officers
                    # Step 1: Click the "More" link under Officers & Directors
                    more_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//div[@id="offDirectors"]//a[contains(text(), "More")]')))
                    driver.execute_script("arguments[0].click();", more_button)
                    logging.info("Clicked on 'More' under Officers & Directors.")
                    time.sleep(5)

                    # Step 2: Wait for the table to appear
                    wait.until(EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'ag-root-wrapper')]")))
                    # Locate the grid rows
                    rows = driver.find_elements(By.XPATH, "//div[contains(@class, 'ag-center-cols-container')]/div[contains(@class, 'ag-row')]")

                    merged_data = []
                    for i, row in enumerate(rows):
                        try:
                            # Each cell is a div with class ag-cell
                            cells = row.find_elements(By.XPATH, ".//div[contains(@class, 'ag-cell')]")
                            if len(cells) >= 2:
                                name = cells[0].text.strip()
                                title = cells[1].text.strip()
                                merged_data.append({
                                    "Name": name if name else "[No Name]",
                                    "Title": title if title else "[No Title]"
                                })

                        except Exception as e:
                            logging.warning(f"Could not process row {i+1}: {e}")

                    # final_output = '; '.join(merged_data)
                    company_data["Key People"] = merged_data
                    # Navigate to the company's LinkedIn "people" page

                    # Step 12: Get LinkedIN URLs of Key People
                    try:
                        wait = WebDriverWait(driver, 10)
                        # Find the span with the text and get its parent anchor tag
                        linkedin_element = driver.find_element(By.CSS_SELECTOR, 'a.spg-link[aria-label="View LinkedIn Professional"]')
                        linkedin_url = linkedin_element.get_attribute('href')
                        parsed_url = urlparse(linkedin_url)
                        clean_path = parsed_url.path.replace('//', '/').rstrip('/')
                        clean_url = urlunparse(parsed_url._replace(path=clean_path, query=''))
                        company_data["LinkedIN URL"] = clean_url
                        driver.execute_script("window.open(arguments[0]);", clean_url)
                        driver.switch_to.window(driver.window_handles[-1])
                    except Exception as e:
                        print("LinkedIn link not found:")
                        company_data["LinkedIN URL"] = None  
                        
                    # After switching to the LinkedIn tab
                    # Step 13: Log in to LinkedIn
                    try:
                        # Wait for login page to load
                        wait.until(EC.presence_of_element_located((By.ID, 'username')))

                        # Enter credentials
                        username_input = driver.find_element(By.ID, 'username')
                        password_input = driver.find_element(By.ID, 'password')

                        username_input.send_keys("ilabarshilia@gmail.com")  # Replace with your LinkedIn email
                        password_input.send_keys("Iambrilliant20")           # Replace with your LinkedIn password

                        # Click login
                        login_button = driver.find_element(By.XPATH, '//button[@type="submit"]')
                        login_button.click()

                        # Wait for login to complete
                        wait.until(EC.presence_of_element_located((By.ID, 'global-nav-search')))
                        print("Logged in successfully.")
                    except Exception as e:
                        print("Login failed:", e)

                    # Step 2: Search each person and extract profile URL
                    linkedin_profiles = {}
                    for person in company_data["Key People"]:
                        name = person["Name"]
                        try:
                            # Locate and use the LinkedIn search bar
                            driver.get(linkedin_url)
                            search_box = wait.until(EC.presence_of_element_located((By.ID, 'people-search-keywords')))
                            search_box.clear()
                            search_box.send_keys(name)
                            search_box.send_keys(Keys.RETURN)
                            time.sleep(5)

                            # Wait for and extract the first profile link
                            profile_link = wait.until(EC.presence_of_element_located((By.XPATH, '//a[contains(@href, "/in/")]')))
                            linkedin_profiles[name] = profile_link.get_attribute('href')
                        except Exception as e:
                            logging.warning(f"Could not find LinkedIn profile for {name}")
                            linkedin_profiles[name] = None

                    company_data["LinkedIN Profiles"] = linkedin_profiles
                    all_company_data.append(company_data)
                    break
            except Exception as e:
                print(f"⚠️ Error processing link: {e}")

        if not match_found:
            logging.info(f"❗ No matching company found for search_term: {search_term} with search_city:{search_city}")
            company_data = {
                "Employer Name": search_term,
                "City": search_city,
                "NAICS Code": None,
                "Industry": None,
                "Description": None,
                "Website URL": None,
                "Headquarters": None,
                "Current_Headcount": None,
                "Percent_Change_in_HC": None,
                "Change_in_HC": None,
                "CEO Name": None,
                "CEO Profile Link": None,
                "LinkedIN URL": None,
                "Key People": None
            }
            all_company_data.append(company_data)
            break

    except Exception as e:
        print(f"❌ Error locating company links: {e}")
        continue

driver.quit()

# Create a DataFrame
df2 = pd.DataFrame([all_company_data])

# Step 1: Explode the 'Key People' column to separate rows
company_data_exploded = df2.explode('Key People')

# Step 2: Extract 'Name' and 'Title' from each dictionary
company_data_exploded['Executive Name'] = company_data_exploded['Key People'].apply(lambda x: x['Name'])
company_data_exploded['Executive Title'] = company_data_exploded['Key People'].apply(lambda x: x['Title'])

# Step 3: Drop the original 'Key People' column
company_data_exploded = company_data_exploded.drop(columns=['Key People'])
# Step 4: Map LinkedIn profiles to each executive
company_data_exploded['LinkedIN Profiles'] = company_data_exploded['Executive Name'].map(linkedin_profiles)
new_col_order = ["Employer Name", "City", "NAICS Code", "Industry", "Description", "Website URL", "Headquarters",
                 "Current_Headcount", "Percent_Change_in_HC", "Change_in_HC", "CEO Name","CEO Profile Link", "LinkedIN URL", 
                 "Executive Name", "Executive Title", "LinkedIN Profiles"]
company_data_exploded = company_data_exploded[new_col_order]
company_data_exploded

2025-07-24 17:28:25 - INFO - Opened Capital IQ login page.


TimeoutException: Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff6587be935+77845]
	GetHandleVerifier [0x0x7ff6587be990+77936]
	(No symbol) [0x0x7ff658579cda]
	(No symbol) [0x0x7ff6585d06aa]
	(No symbol) [0x0x7ff6585d095c]
	(No symbol) [0x0x7ff658623d07]
	(No symbol) [0x0x7ff6585f890f]
	(No symbol) [0x0x7ff658620b07]
	(No symbol) [0x0x7ff6585f86a3]
	(No symbol) [0x0x7ff6585c1791]
	(No symbol) [0x0x7ff6585c2523]
	GetHandleVerifier [0x0x7ff658a9684d+3059501]
	GetHandleVerifier [0x0x7ff658a90c0d+3035885]
	GetHandleVerifier [0x0x7ff658ab0400+3164896]
	GetHandleVerifier [0x0x7ff6587d8c3e+185118]
	GetHandleVerifier [0x0x7ff6587e054f+216111]
	GetHandleVerifier [0x0x7ff6587c72e4+113092]
	GetHandleVerifier [0x0x7ff6587c7499+113529]
	GetHandleVerifier [0x0x7ff6587ae298+10616]
	BaseThreadInitThunk [0x0x7ff80c5de8d7+23]
	RtlUserThreadStart [0x0x7ff80d2fc34c+44]
